In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
        .master("local") \
        .appName("MyApp") \
        .getOrCreate()

# Verify Spark Session
print(spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/01 09:46:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.1


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    IntegerType, LongType, DoubleType, FloatType,
    ShortType, DecimalType, StringType
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator

from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor

# ===================================================================
#                 FEATURE DETECTION
# ===================================================================
def detect_features(df, target_col):
    numeric_types = (
        IntegerType, LongType, DoubleType,
        FloatType, ShortType, DecimalType
    )

    numeric_features = []
    categorical_features = []

    for field in df.schema.fields:
        if field.name == target_col:
            continue

        dtype = field.dataType

        if isinstance(dtype, numeric_types):
            numeric_features.append(field.name)
        elif isinstance(dtype, StringType):
            categorical_features.append(field.name)

    print("Auto-detected numeric features:", numeric_features)
    print("Auto-detected categorical features:", categorical_features)

    return numeric_features, categorical_features

def filter_high_cardinality_features(df, categorical_features, max_categories=200):
    """
    Removes categorical features that have more than `max_categories` distinct values.
    """
    filtered = []
    print("\n[INFO] Checking categorical feature cardinality...")

    for col in categorical_features:
        unique_count = df.select(col).distinct().count()
        print(f"[INFO] {col}: {unique_count} unique values")

        if unique_count <= max_categories:
            filtered.append(col)
        else:
            print(f"[WARN] Dropping feature '{col}' due to high cardinality ({unique_count}).")

    print(f"[INFO] Remaining categorical features: {filtered}")
    return filtered


# ===================================================================
#                 DROP NULLS
# ===================================================================
def drop_null_rows(df, numeric_features, categorical_features, target_col):
    """
    Drops only rows where target is NULL,
    fills missing numeric features with 0,
    fills missing categorical features with 'unknown'.
    """

    print("[INFO] Cleaning NULL values...")

    before = df.count()

    df_clean = df.dropna(subset=[target_col])

    if numeric_features:
        df_clean = df_clean.fillna(0, subset=numeric_features)

    if categorical_features:
        df_clean = df_clean.fillna("unknown", subset=categorical_features)

    after = df_clean.count()

    print(f"[INFO] Rows before cleaning: {before}")
    print(f"[INFO] Rows after cleaning:  {after}")
    print(f"[INFO] Dropped: {before - after} rows with NULL target.")

    return df_clean

# ===================================================================
#                 BUILD PIPELINE
# ===================================================================
def build_preprocessing_pipeline(numeric_features, categorical_features):
    stages = []

    if numeric_features:
        assembler_num = VectorAssembler(
            inputCols=numeric_features,
            outputCol="numeric_vector"
        )
        stages.append(assembler_num)

        scaler = StandardScaler(
            inputCol="numeric_vector",
            outputCol="numeric_scaled",
            withStd=True,
            withMean=False
        )
        stages.append(scaler)

    # Encode categorical
    for col in categorical_features:
        stages.append(
            StringIndexer(
                inputCol=col,
                outputCol=f"{col}_idx",
                handleInvalid="keep"
            )
        )

    # Combine into a feature vector
    feature_cols = ["numeric_scaled"] + [f"{c}_idx" for c in categorical_features]

    stages.append(
        VectorAssembler(
            inputCols=feature_cols,
            outputCol="features"
        )
    )

    return stages


# ===================================================================
#                 MODEL TYPE CHECKING
# ===================================================================
def is_regression_model(algorithm):
    return isinstance(algorithm, (LinearRegression, RandomForestRegressor, GBTRegressor))


def is_classification_model(algorithm):
    return isinstance(algorithm, (LogisticRegression, RandomForestClassifier, GBTClassifier))


# ===================================================================
#                 MAIN PIPELINE LOGIC
# ===================================================================

    
def run_training(df, target_col, algorithm, params=None):
    # Detect
    numeric_features, categorical_features = detect_features(df, target_col)

    # Clean
    df = drop_null_rows(df, numeric_features, categorical_features, target_col)

    categorical_features = filter_high_cardinality_features(
        df, categorical_features, max_categories=50
    )

    # Pipeline
    preprocessing_stages = build_preprocessing_pipeline(
        numeric_features=numeric_features,
        categorical_features=categorical_features
    )

    if params:
        algorithm = algorithm.setParams(**params)

    pipeline = Pipeline(stages=preprocessing_stages + [algorithm])

    # Split
    train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

    # Train
    model = pipeline.fit(train_df)
    predictions = model.transform(test_df)

    # Evaluation
    if is_classification_model(algorithm):
        evaluator = MulticlassClassificationEvaluator(
            labelCol=target_col,
            predictionCol="prediction",
            metricName="accuracy"
        )
        metric = evaluator.evaluate(predictions)
        print(f"[RESULT] Classification accuracy: {metric}")

    elif is_regression_model(algorithm):
        evaluator = RegressionEvaluator(
        labelCol=target_col,
        predictionCol="prediction"
        )

        rmse = evaluator.setMetricName("rmse").evaluate(predictions)
        mse  = evaluator.setMetricName("mse").evaluate(predictions)
        mae  = evaluator.setMetricName("mae").evaluate(predictions)
        r2   = evaluator.setMetricName("r2").evaluate(predictions)

        print("=== REGRESSION METRICS ===")
        print(f"RMSE: {rmse}")
        print(f"MSE : {mse}")
        print(f"MAE : {mae}")
        print(f"R2  : {r2}")

    return model

In [ ]:
# ANSI COLORS
GREEN = "\033[92m"
BLUE = "\033[94m"
YELLOW = "\033[93m"
CYAN = "\033[96m"
RED = "\033[91m"
BOLD = "\033[1m"
RESET = "\033[0m"

def banner(text):
    print(f"\n{BOLD}{BLUE}{'=' * 80}\n{text.center(80)}\n{'=' * 80}{RESET}\n")

def section(text):
    print(f"\n{BOLD}{CYAN}--- {text} ---{RESET}")

def info(text):
    print(f"{GREEN}[INFO]{RESET} {text}")

def warn(text):
    print(f"{YELLOW}[WARN]{RESET} {text}")

def error(text):
    print(f"{RED}[ERROR]{RESET} {text}")

# ===================================================================
#                          SPARK JOB START
# ===================================================================
banner("SPARK JOB STARTED")

section("INITIALIZING SPARK SESSION")
info("Creating SparkSession...")

spark = SparkSession.builder \
    .appName("internet_training") \
    .getOrCreate()

# ===================================================================
#                        LOAD CSV INTO DATAFRAME
# ===================================================================
section("LOADING CSV DATASET")

csv_path = "/kaggle/input/internet-dataset/fbd_us_with_satellite_dec2021_v1.csv"
info(f"CSV path: {csv_path}")

internet_df = spark.read.csv(
    csv_path,
    header=True,
    inferSchema=True
)

info("CSV successfully loaded.")

section("DATAFRAME SCHEMA")
internet_df.printSchema()

section("FIRST 5 ROWS")
internet_df.show(5)

# ===================================================================
#                         TARGET & SAMPLING
# ===================================================================
section("TARGET COLUMN")
target_col = "MaxAdDown"
info(f"Target column set to: {target_col}")

section("APPLYING 1% SAMPLING")
before_sample = internet_df.count()
internet_df = internet_df.sample(withReplacement=False, fraction=0.01, seed=42)
after_sample = internet_df.count()

info(f"Rows before sampling:  {before_sample}")
info(f"Rows after sampling:   {after_sample}")
info(f"Sampling ratio actual: {after_sample / before_sample:.4f}")

# ===================================================================
#                       MODELS TO TRAIN (LOOP)
# ===================================================================
section("CONFIGURING MODELS")

models_to_train = [
    # ("RandomForestRegressor", RandomForestRegressor(
    #     labelCol=target_col,
    #     maxDepth=12,
    #     numTrees=50,
    #     maxBins=32
    # )),
    ("LinearRegression", LinearRegression(
        labelCol=target_col,
        maxIter=200,
        regParam=0.2
    )),
]

banner("TRAINING PIPELINE STARTED")

import time

for model_name, algorithm in models_to_train:
    banner(f"TRAINING MODEL: {model_name}")

    info(f"Algorithm params: {algorithm.extractParamMap()}")

    start_time = time.time()

    trained_model = run_training(
        df=internet_df,
        target_col=target_col,
        algorithm=algorithm
    )

    duration = time.time() - start_time
    info(f"Training duration for {model_name}: {duration:.2f} seconds")

    banner(f"FINISHED MODEL: {model_name}")


banner("TRAINING PIPELINE COMPLETED")

section("STOPPING SPARK")
spark.stop()

banner("SPARK JOB FINISHED")


                               SPARK JOB STARTED                                


--- INITIALIZING SPARK SESSION ---
[INFO] Creating SparkSession...


25/12/01 09:46:51 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.



--- LOADING CSV DATASET ---
[INFO] CSV path: /kaggle/input/internet-dataset/fbd_us_with_satellite_dec2021_v1.csv


[INFO] CSV successfully loaded.

--- DATAFRAME SCHEMA ---
root
 |-- LogRecNo: integer (nullable = true)
 |-- Provider_Id: integer (nullable = true)
 |-- FRN: integer (nullable = true)
 |-- ProviderName: string (nullable = true)
 |-- DBAName: string (nullable = true)
 |-- HoldingCompanyName: string (nullable = true)
 |-- HocoNum: integer (nullable = true)
 |-- HocoFinal: string (nullable = true)
 |-- StateAbbr: string (nullable = true)
 |-- BlockCode: long (nullable = true)
 |-- TechCode: integer (nullable = true)
 |-- Consumer: integer (nullable = true)
 |-- MaxAdDown: double (nullable = true)
 |-- MaxAdUp: double (nullable = true)
 |-- Business: integer (nullable = true)


--- FIRST 5 ROWS ---
+--------+-----------+-------+-------------+--------------------+------------------+-------+---------------+---------+---------------+--------+--------+---------+-------+--------+
|LogRecNo|Provider_Id|    FRN| ProviderName|             DBAName|HoldingCompanyName|HocoNum|      HocoFinal|StateAbb